# 类继承关系
```mermaid
classDiagram
    %% 元类层次结构
    class Accessor
    class Vbt_SRAccessor
    class DirNamesMixin
    class GenericSRAccessor
    class GenericDFAccessor
    class Vbt_DFAccessor
    
    %% 继承关系
    GenericSRAccessor <|-- Vbt_SRAccessor
    DirNamesMixin <|-- Vbt_SRAccessor

    DirNamesMixin <|-- Vbt_DFAccessor
    GenericDFAccessor <|-- Vbt_DFAccessor
```

# class Accessor

```python
class Accessor:

    def __init__(self, name: str, accessor: tp.Type[AccessorT]) -> None:
        self._name = name
        self._accessor = accessor

    def __get__(self, obj: ParentAccessorT, cls: DirNamesMixin) -> AccessorT:
        if obj is None:
            return self._accessor
        if isinstance(obj, (pd.Series, pd.DataFrame)):
            accessor_obj = self._accessor(obj)
        elif isinstance(obj, Configured):
            accessor_obj = obj.replace(cls_=self._accessor)
        else:
            accessor_obj = self._accessor(obj.obj)
        return accessor_obj
```

# register_accessor
用于装饰一个类：
```python
@register_accessor('my_accessor', Series/DataFrame)
class MyAccessor: ...
```
- 即
    ```python
    register_accessor('my_accessor', Series/DataFrame)(MyAccessor)
    ```
- 相当于给 Series/DataFrame 增加了一个 Accessor 类型描述符
    ```python
    my_accessor = Accessor(my_accessor, MyAccessor)
    ```
  
于是对于一个 Series/DataFrame 类型的实例 `obj`，`obj.my_accessor` 会发生
- `my_accessor.__get__(obj, type(obj))——>my_accessor._accessor(obj) = MyAccessor(obj)`

即 `obj.my_accessor.*` 等价于 ` MyAccessor(obj).*`

## 源码

```python
def register_accessor(name: str, cls: tp.Type[DirNamesMixin]) -> tp.Callable:

    def decorator(accessor: tp.Type[AccessorT]) -> tp.Type[AccessorT]:
        if hasattr(cls, name):
            warnings.warn(
                f"registration of accessor {repr(accessor)} under name "
                f"{repr(name)} for type {repr(cls)} is overriding a preexisting "
                f"attribute with the same name.",
                UserWarning,
                stacklevel=2,
            )
        setattr(cls, name, Accessor(name, accessor))
        cls._accessors.add(name)
        return accessor

    return decorator
```

## 例子

In [ ]:
import pandas as pd
from vectorbt.root_accessors import register_accessor

@register_accessor('my_custom_accessor', pd.Series)
class MyCustomAccessor:
    def __init__(self, obj):
        self._obj = obj
    
    def my_method(self):
        return self._obj.sum()

# 使用注册的访问器
series = pd.Series([1, 2, 3, 4, 5])
result = series.my_custom_accessor.my_method()
print(result)

## register_series_accessor

```python
def register_series_accessor(name: str) -> tp.Callable:
    return register_accessor(name, pd.Series)
```

### class Vbt_SRAccessor(DirNamesMixin, GenericSRAccessor)
对于一个 `Series` 的实例 `obj`，于是 `obj.vbt` 就相当于
- `Vbt_SRAccessor(obj)`

```python
@register_series_accessor("vbt")
class Vbt_SRAccessor(DirNamesMixin, GenericSRAccessor):

    def __init__(self, obj: tp.Series, **kwargs) -> None:
        self._obj = obj

        DirNamesMixin.__init__(self)
        GenericSRAccessor.__init__(self, obj, **kwargs)
```

#### 例子

In [ ]:
import pandas as pd
import numpy as np

# 创建示例时间序列数据
dates = pd.date_range('2023-01-01', periods=100, freq='D')
prices = pd.Series(np.random.randn(100).cumsum() + 100, index=dates, name='price')

# 基本统计分析
stats = prices.vbt.describe()
print("基本统计信息:", stats)

# 滚动窗口分析
ma_5 = prices.vbt.rolling_mean(window=5)      # 5日移动平均
ma_20 = prices.vbt.rolling_mean(window=20)    # 20日移动平均
volatility = prices.vbt.rolling_std(window=20)  # 20日波动率

# 数据变换
returns = prices.vbt.pct_change()             # 收益率
normalized = prices.vbt.normalize()           # 标准化

# 正确的填充方法
filled_forward = prices.vbt.ffill()           # 前向填充
filled_backward = prices.vbt.bfill()          # 后向填充
filled_value = prices.vbt.fillna(value=100)   # 用固定值填充

# 绘图可视化
fig = prices.vbt.plot(title='价格走势')
fig.show()

# 使用专用访问器
# 信号分析（需要布尔序列）
signals = (prices > ma_20).vbt.signals
print("信号统计:", signals.describe())

# 收益率分析
returns_analysis = returns.vbt.returns
print("收益率统计:", returns_analysis.describe())

## register_dataframe_accessor

```python
def register_dataframe_accessor(name: str) -> tp.Callable:
    return register_accessor(name, pd.DataFrame)
```

### class Vbt_DFAccessor(DirNamesMixin, GenericDFAccessor)

```python
@register_dataframe_accessor("vbt")
class Vbt_DFAccessor(DirNamesMixin, GenericDFAccessor):

    def __init__(self, obj: tp.Frame, **kwargs) -> None:
        self._obj = obj

        DirNamesMixin.__init__(self)
        GenericDFAccessor.__init__(self, obj, **kwargs)
```

#### 例子

In [ ]:
import pandas as pd
import numpy as np

# 创建示例多资产价格数据
dates = pd.date_range('2023-01-01', periods=100, freq='D')
np.random.seed(42)
prices = pd.DataFrame({
    'AAPL': np.random.randn(100).cumsum() + 150,
    'GOOGL': np.random.randn(100).cumsum() + 2500,
    'MSFT': np.random.randn(100).cumsum() + 300,
    'TSLA': np.random.randn(100).cumsum() + 200
}, index=dates)

# 基本统计分析
stats = prices.vbt.describe()
print("多资产统计信息:", stats)

# 相关性分析 - 使用 pandas 原生方法
correlation = prices.corr()
print("价格相关性矩阵:")
print(correlation)

# 收益率相关性
returns = prices.pct_change().dropna()
returns_correlation = returns.corr()
print("\n收益率相关性矩阵:")
print(returns_correlation)

# 滚动窗口分析
ma_20 = prices.vbt.rolling_mean(window=20)    # 20日移动平均
volatility = prices.vbt.rolling_std(window=20)  # 20日波动率

# 数据变换
normalized = prices.vbt.normalize()           # 标准化处理

# 绘图可视化
fig = prices.vbt.plot(title='多资产价格走势')
fig.show()

# 热力图
heatmap = prices.vbt.heatmap(title='价格热力图')
heatmap.show()

# 使用专用访问器
# 收益率分析
returns_analysis = returns.vbt.returns
print("\n收益率统计:", returns_analysis.describe())

# 组合分析
portfolio_returns = returns.mean(axis=1)  # 等权重组合
portfolio_stats = portfolio_returns.vbt.returns.describe()
print("\n组合统计:", portfolio_stats)

## register_series_vbt_accessor

```python
def register_series_vbt_accessor(name: str, parent: tp.Type[DirNamesMixin] = Vbt_SRAccessor) -> tp.Callable:

    return register_accessor(name, parent)
```

## register_dataframe_vbt_accessor

```python
def register_dataframe_vbt_accessor(name: str, parent: tp.Type[DirNamesMixin] = Vbt_DFAccessor) -> tp.Callable:

    return register_accessor(name, parent)
```